# Notebook Thử Nghiệm Triển Khai RAG (Retrieval-Augmented Generation) với Dữ liệu Thực tế

Notebook này được thiết kế để thử nghiệm luồng **RAG** sử dụng dữ liệu thực tế được trích xuất từ video bài thuyết trình TED Talk (nằm trong thư mục `demo_data`).

### Dữ liệu thực tế được nạp:
1. **ASR Transcript (`youtube_clip_extracted_transcript.txt`)**: Nội dung lời nói của diễn giả kèm theo mốc thời gian chi tiết.
2. **Visual Results (`visual_result.json`)**: Các phân đoạn phân cảnh (scenes) và ảnh keyframe trích xuất từ video.

### Các bước thực hiện:
* Đọc và phân tích (parse) file transcript thực tế.
* Khởi tạo kết nối tới ChromaDB (Local/Cloud).
* Đưa dữ liệu thực tế vào DB để tạo chỉ mục vector.
* Kiểm thử truy vấn Vector (Similarity Search) với câu hỏi thực tế về nội dung bài nói (Ví dụ: "Dogma của xã hội phương Tây là gì?", "Tại sao nhiều lựa chọn lại không tốt?").
* Tạo câu trả lời RAG qua mô hình ngôn ngữ lớn (Llama-3 trên Groq).

In [ ]:
import os
import sys
import json
from pathlib import Path

# Thiết lập sys.path để import các module cấu hình của hệ thống
project_root = Path("../../").resolve()
sys.path.append(str(project_root))
sys.path.append(str(project_root / "backend"))

# Đọc cấu hình từ file backend/.env
try:
    from dotenv import load_dotenv
    load_dotenv(project_root / "backend" / ".env")
except ImportError:
    # Dự phòng nếu môi trường chưa cài python-dotenv
    env_path = project_root / "backend" / ".env"
    if env_path.exists():
        for line in env_path.read_text(encoding="utf-8").splitlines():
            if line.strip() and not line.startswith("#"):
                if "=" in line:
                    key, val = line.split("=", 1)
                    os.environ[key.strip()] = val.strip().strip('"').strip("'")

print("✅ Cấu hình môi trường đã tải thành công!")
print(f"- APP_ENV: {os.getenv('APP_ENV')}")
print(f"- CHROMADB_HOST: {os.getenv('CHROMADB_HOST')}")
print(f"- Có GROQ_API_KEY: {bool(os.getenv('GROQ_API_KEY'))}")

## Bước 1: Khởi tạo kết nối đến Vector DB (ChromaDB)
Chúng ta sẽ khởi tạo Client kết nối đến ChromaDB Cloud (nếu có API Key) hoặc HTTP Local Client.

In [ ]:
import chromadb

chroma_host = os.getenv("CHROMADB_HOST", "localhost")
chroma_port = int(os.getenv("CHROMADB_PORT", "8000"))
chroma_ssl = os.getenv("CHROMADB_SSL", "False").lower() in ("true", "1", "yes")
chroma_api_key = os.getenv("CHROMADB_API_KEY", "")
chroma_tenant = os.getenv("CHROMADB_TENANT", "default_tenant")
chroma_database = os.getenv("CHROMADB_DATABASE", "default_database")
collection_name = "rag_real_data_experiment"

# Kết nối dựa theo biến môi trường
if chroma_api_key:
    print(f"🔗 Đang kết nối tới ChromaDB Cloud tại {chroma_host}:{chroma_port}...")
    chroma_client = chromadb.CloudClient(
        tenant=chroma_tenant,
        database=chroma_database,
        api_key=chroma_api_key,
        cloud_host=chroma_host,
        cloud_port=chroma_port,
        enable_ssl=chroma_ssl
    )
else:
    print(f"🔗 Đang kết nối tới ChromaDB Local tại http://{chroma_host}:{chroma_port}...")
    chroma_client = chromadb.HttpClient(
        host=chroma_host,
        port=chroma_port,
        ssl=chroma_ssl,
        tenant=chroma_tenant,
        database=chroma_database
    )

# Xóa collection trùng nếu đã tồn tại để tránh xung đột dữ liệu cũ
try:
    chroma_client.delete_collection(name=collection_name)
    print(f"🗑️ Đã dọn dẹp collection cũ: '{collection_name}'")
except Exception:
    pass

# Tạo một collection mới
# Lưu ý: ChromaDB Client sẽ tự động sinh Embedding qua mô hình mặc định 'all-MiniLM-L6-v2'
collection = chroma_client.create_collection(name=collection_name)
print(f"✨ Đã tạo mới collection thành công: '{collection_name}'")

## Bước 2: Đọc và phân tích (Parse) dữ liệu bài giảng thực tế
Chúng ta đọc dữ liệu từ file `demo_data/results/youtube_clip_extracted_transcript.txt` và `demo_data/results/visual_result.json`.

In [ ]:
def to_seconds(t_str):
    t_str = t_str.strip()
    parts = t_str.split(":")
    if len(parts) == 3: # HH:MM:SS
        return int(parts[0])*3600 + int(parts[1])*60 + float(parts[2])
    elif len(parts) == 2: # MM:SS
        return int(parts[0])*60 + float(parts[1])
    return float(t_str)

def load_real_data():
    documents = []
    metadatas = []
    ids = []
    chunk_counter = 0

    # 1. Đọc và parse Transcript thực tế
    transcript_path = Path("demo_data/results/youtube_clip_extracted_transcript.txt")
    if transcript_path.exists():
        print(f"Reading transcript from {transcript_path}...")
        with open(transcript_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                # Định dạng dòng: [00:25 - 00:30]: I want to talk...
                if line.startswith("[") and "]:" in line:
                    time_part, text_part = line.split("]:", 1)
                    time_range = time_part.strip("[").strip()
                    start_str, end_str = time_range.split("-", 1)
                    
                    start_sec = to_seconds(start_str)
                    end_sec = to_seconds(end_str)
                    text = text_part.strip()
                    
                    # Bỏ các đoạn im lặng/nhạc nền không mang thông tin chính
                    if text == "[Nhạc nền / Im lặng]" or not text:
                        continue
                    
                    doc_text = f"[Speech][{time_range}]: {text}"
                    documents.append(doc_text)
                    metadatas.append({
                        "start": start_sec,
                        "end": end_sec,
                        "type": "asr"
                    })
                    ids.append(f"real_chunk_{chunk_counter}")
                    chunk_counter += 1
    else:
        print("⚠️ Không tìm thấy file transcript thực tế!")

    # 2. Đọc và parse Visual Scenes thực tế
    visual_path = Path("demo_data/results/visual_result.json")
    if visual_path.exists():
        print(f"Reading visual keyframes from {visual_path}...")
        with open(visual_path, "r", encoding="utf-8") as f:
            v_data = json.load(f)
            for idx, scene in enumerate(v_data.get("scenes", [])):
                start_sec = scene.get("start_seconds", 0.0)
                end_sec = scene.get("end_seconds", 0.0)
                time_range = f"{scene.get('start_timecode', '00:00:00')} - {scene.get('end_timecode', '00:00:00')}"
                caption = scene.get("caption", "").strip()
                
                # Chỉ nạp nếu có caption thực sự hữu ích
                if caption and caption != "A representative frame extracted from the lecture video.":
                    doc_text = f"[Visual][{time_range}]: {caption}"
                    documents.append(doc_text)
                    metadatas.append({
                        "start": start_sec,
                        "end": end_sec,
                        "type": "ocr"
                    })
                    ids.append(f"real_chunk_{chunk_counter}")
                    chunk_counter += 1
    else:
        print("⚠️ Không tìm thấy file visual_result.json!")

    return documents, metadatas, ids

# Chạy load data thực tế
documents, metadatas, ids = load_real_data()
print(f"\n✅ Đã chuẩn bị xong {len(documents)} chunks từ dữ liệu thực tế!")
# In thử 5 chunks đầu tiên
for doc in documents[:5]:
    print(f"- {doc}")

## Bước 3: Nạp dữ liệu thực tế vào DB để tạo chỉ mục vector

In [ ]:
if documents:
    collection.add(
        documents=documents,
        metadatas=metadatas,
        ids=ids
)
    print("📥 Nạp dữ liệu thực tế vào vector database thành công!")
else:
    print("❌ Không có dữ liệu để nạp. Vui lòng kiểm tra lại đường dẫn file.")

## Bước 4: Tìm kiếm Vector trên dữ liệu thực tế (Similarity Search)
Chúng ta sẽ hỏi các câu hỏi thực tế về chủ đề **"Sự nghịch lý của sự lựa chọn" (The Paradox of Choice)** dựa trên lời thoại của diễn giả Barry Schwartz.

In [ ]:
def query_real_vector_db(user_query, limit=3):
    results = collection.query(
        query_texts=[user_query],
        n_results=limit
    )
    
    print(f"🔍 Câu hỏi tìm kiếm: '{user_query}'")
    print("=" * 60)
    
    retrieved_contexts = []
    if results and "documents" in results and results["documents"]:
        docs = results["documents"][0]
        distances = results["distances"][0]
        
        for i, (doc, dist) in enumerate(zip(docs, distances)):
            print(f"🎯 Đoạn liên quan {i+1} [Độ lệch Cosine: {dist:.4f}]:")
            print(f"   {doc}")
            print()
            retrieved_contexts.append(doc)
    return retrieved_contexts

# Câu hỏi 1: Về official dogma (giáo điều chính thức) của phương Tây
q1 = "What is the official dogma of Western industrial societies?"
context_q1 = query_real_vector_db(q1, limit=3)

print("-" * 80)

# Câu hỏi 2: Về số lượng salad dressing trong siêu thị
q2 = "How many salad dressings are in the supermarket?"
context_q2 = query_real_vector_db(q2, limit=3)

## Bước 5: Gọi Groq LLM để trả lời RAG dựa trên dữ liệu thực tế

In [ ]:
from openai import OpenAI

groq_api_key = os.getenv("GROQ_API_KEY", "")
groq_model = os.getenv("GROQ_MODEL", "llama-3.1-8b-instant")

if not groq_api_key:
    print("❌ LỖI: GROQ_API_KEY chưa được đặt trong .env. Vui lòng cấu hình key để chạy phần sinh câu trả lời RAG.")
else:
    # Khởi tạo client gọi Groq API
    client = OpenAI(
        base_url="https://api.groq.com/openai/v1",
        api_key=groq_api_key
    )

    def generate_rag_response(question, contexts):
        context_str = "\n".join([f"- {chunk}" for chunk in contexts])

        system_prompt = (
            "You are an expert AI teaching assistant. Answer the student's question "
            "based strictly on the provided lecture transcript/slide contexts below.\n\n"
            "Rules:\n"
            "- Be precise, objective, and write in the same language as the student's question (or English since the lecture is in English).\n"
            "- Rely ONLY on the provided context. If the context does not contain enough info, state clearly that it is not mentioned."
        )

        user_prompt = f"""Context from the lecture:
{context_str}

Student Question: {question}

Answer:"""

        print(f"🤖 Generating RAG answer using '{groq_model}'...")
        completion = client.chat.completions.create(
            model=groq_model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.2
        )
        
        answer = completion.choices[0].message.content
        print("\n=================== RAG RESPONSE ===================")
        print(f"Q: {question}")
        print("-" * 60)
        print(answer)
        print("====================================================\n")
        return answer

    # Gọi RAG cho Câu hỏi 1
    generate_rag_response(q1, context_q1)

    # Gọi RAG cho Câu hỏi 2
    generate_rag_response(q2, context_q2)

## Bước 6: Dọn dẹp tài nguyên
Xóa collection thử nghiệm thực tế khỏi database.

In [ ]:
try:
    chroma_client.delete_collection(name=collection_name)
    print(f"🗑️ Đã dọn dẹp collection thử nghiệm thực tế: '{collection_name}'")
except Exception as e:
    print(f"Lỗi dọn dẹp: {e}")